In [ ]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageEnhance, ImageDraw
from transformers import CLIPProcessor, CLIPModel
import cv2
from pathlib import Path
import logging
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import sys
sys.path.insert(0,"../src")
from mai_grounding_agent import MAIGroundingAgent
from utils import draw_clicks_on_image, extract_click_coordinates
sys.path.append('../MAI UI/MAI-UI/evaluation/grounding')
from eval_local_uivision import load_dataset

In [ ]:
# Model configuration
BASE_CLIP = "../clip-vit-base-patch32"
CKPT_PATH = "../clip_model_weights.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CLAHE = True
print("Loading CLIP...")

model = CLIPModel.from_pretrained(BASE_CLIP)
processor = CLIPProcessor.from_pretrained(BASE_CLIP)

if os.path.exists(CKPT_PATH):
    state = torch.load(CKPT_PATH, map_location="cpu")
    if "model" in state:
        state = state["model"]
    model.load_state_dict(state, strict=False)

model = model.to(DEVICE).eval()
torch.set_grad_enabled(False)

def pad_bbox_gt(bbox, pad, W, H):
    x1, y1, x2, y2 = bbox
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(W - 1, x2 + pad)
    y2 = min(H - 1, y2 + pad)
    return [x1, y1, x2, y2]

def hybrid_vector_field_grid(image_path, instruction, num_grid=36, sigma=0.4, batch_size=8):
    img = Image.open(image_path).convert("RGB")
    W, H = img.size
    pw, ph = W // num_grid, H // num_grid
    patches = [img.crop((gx*pw, gy*ph, (gx+1)*pw, (gy+1)*ph)) for gy in range(num_grid) for gx in range(num_grid)]

    text_inputs = processor(text=instruction, return_tensors="pt").to(DEVICE)
    text_emb = model.get_text_features(**text_inputs)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
    text_emb = text_emb.cpu().numpy()

    feats = []
    for i in range(0, len(patches), batch_size):
        inputs = processor(images=patches[i:i+batch_size], return_tensors="pt").to(DEVICE)
        emb = model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        feats.append(emb.cpu().numpy())
    patch_embs = np.concatenate(feats, axis=0)

    cos_sim = cosine_similarity(patch_embs, text_emb).flatten()
    beta = 3.0
    gates = 1 / (1 + np.exp(-beta * (cos_sim - 0.5)))
    gated = patch_embs * gates[:, None]

    N = gated.shape[0]
    F_field = np.zeros_like(gated)
    for i in range(N):
        diff = gated - gated[i]
        dist = np.linalg.norm(diff, axis=1)
        w = np.exp(-dist**2 / sigma**2) * gates
        F_field[i] = np.sum(w[:, None] * diff, axis=0)
    F_field /= (np.linalg.norm(F_field, axis=1, keepdims=True) + 1e-8)

    div = np.zeros(N)
    k = min(8, N-1)
    for i in range(N):
        diff = F_field - F_field[i]
        idx = np.argsort(np.linalg.norm(diff, axis=1))[:k]
        grad = diff[idx].mean(axis=0)
        div[i] = np.dot(grad, F_field[i])

    score = np.log1p(np.maximum(0, -div)) * cos_sim * gates
    score = (score - score.min()) / (score.max() - score.min() + 1e-8)
    return score.reshape(num_grid, num_grid), (H, W)

def apply_spotlight_on_image(image_path, mask_full):
    mask_blur = cv2.GaussianBlur(mask_full, (0,0), 8)
    mask3 = np.stack([mask_blur]*3, axis=-1)

    img_np = np.array(Image.open(image_path).convert("RGB")).astype(np.float32)/255.0
    dark_np = img_np * 0.07

    bright = Image.fromarray((img_np*255).astype(np.uint8))
    bright = ImageEnhance.Brightness(bright).enhance(1.25)
    bright = ImageEnhance.Sharpness(bright).enhance(2.5)
    enhanced_np = np.array(bright).astype(np.float32)/255.0

    if USE_CLAHE:
        lab = cv2.cvtColor((enhanced_np*255).astype(np.uint8), cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l2 = clahe.apply(l)
        l_mix = (1-mask_blur)*l + mask_blur*l2
        lab = cv2.merge([l_mix.astype(np.uint8), a, b])
        enhanced_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)/255.0

    final = dark_np*(1-mask3) + enhanced_np*mask3
    return np.clip(final, 0, 1)


def spotlight_crop_roi(image_path, instruction, out_path=None):
    grid_scores, (H,W) = hybrid_vector_field_grid(image_path, instruction, 36)

    binary = (grid_scores>=0.15).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(binary)

    regions=[]
    for l in range(1,num_labels):
        mask=(labels==l)
        area=mask.sum()
        if area<3:
            continue
        score=grid_scores[mask].mean()*area
        regions.append((score,l))

    regions=sorted(regions, reverse=True)[:4]

    patch_mask=np.zeros_like(grid_scores,dtype=np.float32)
    for _,l in regions:
        patch_mask += (labels==l)

    patch_mask=np.clip(patch_mask,0,1)

    t=torch.tensor(patch_mask)[None,None]
    mask_full=F.interpolate(t, size=(H,W), mode="bicubic", align_corners=False).squeeze().numpy()
    mask_full=(mask_full-mask_full.min())/(mask_full.max()-mask_full.min()+1e-8)

    spotlight_np = apply_spotlight_on_image(image_path, mask_full)
    spotlight_img = Image.fromarray((spotlight_np*255).astype(np.uint8))

    ys,xs=np.where(mask_full>0.25)
    if len(xs)==0:
        return None, None, None, None, None
    CROP_PAD = 8
    x1,x2=xs.min(), xs.max()
    y1,y2=ys.min(), ys.max()

    x1,y1=max(0,x1-CROP_PAD), max(0,y1-CROP_PAD)
    x2,y2=min(W-1,x2+CROP_PAD), min(H-1,y2+CROP_PAD)

    crop = spotlight_img.crop((x1,y1,x2,y2))
    crop_original_size = crop.copy()

    crop_area = (x2-x1)*(y2-y1)
    full_area = W*H
    crop_ratio = crop_area / full_area

    if crop_ratio > 0.95:
        crop_resized = crop
        effective_scale = 1.0
    else:
        crop_resized, effective_scale = resize_roi_keep_ratio(crop, (W,H))

    if out_path:
        crop_resized.save(out_path)

    return (x1,y1,x2,y2), crop_resized, crop_original_size, mask_full, effective_scale

def point_in_bbox(px, py, bbox):
    x1,y1,x2,y2 = bbox
    return x1<=px<=x2 and y1<=py<=y2

def remap_click(px_crop, py_crop, bbox_crop):
    x1,y1,_,_ = bbox_crop
    return px_crop + x1, py_crop + y1

def resize_roi_keep_ratio(roi_img, full_size):
    fw, fh = full_size
    rw, rh = roi_img.size

    scale = min(fw / rw, fh / rh)

    new_w = int(round(rw * scale))
    new_h = int(round(rh * scale))

    resized = roi_img.resize((new_w, new_h), Image.BICUBIC)
    return resized, scale

JSON_PATH = ".../MAI UI/MAI-UI/evaluation/grounding/data/UI_Vision_data"
IMAGE_ROOT = Path(".../ui-vision/images")

VIS_DIR = Path(".../spotlight-svf/results/vis_uivision")
VIS_DIR.mkdir(exist_ok=True, parents=True)
OUTPUT_PATH = Path(".../spotlight-svf/ui_vision.json")

dataset = load_dataset(JSON_PATH)

agent = MAIGroundingAgent(
    llm_base_url="http://localhost:8000/v1",
    model_name="MAI-UI-8B",
    runtime_conf={"history_n":3,"temperature":0.0,"top_k":-1,"top_p":1.0,"max_tokens":2048},
)

In [ ]:
results=[]
correct=0
SAVE_EVERY=200
GT_PAD = 0.01
def safe_save():
    summary={
        "total": int(len(results)),
        "correct": int(correct),
        "accuracy": float(correct/len(results)) if results else 0.0
    }
    with open(OUTPUT_PATH,"w") as f:
        json.dump({"summary":summary,"results":results},f,indent=2)

for sample in tqdm(dataset, desc="Evaluating", disable=True):

    sample_id=str(sample["id"])
    instruction=str(sample["instruction"])
    image_path=IMAGE_ROOT/sample["img_filename"]
    # image_path=IMAGE_ROOT/sample["image_path"]
    bbox_gt=[float(x) for x in sample["bbox"]]

    if not image_path.exists():
        continue

    orig_img_for_size = Image.open(image_path)
    W, H = orig_img_for_size.size
    bbox_gt_padded = pad_bbox_gt(bbox_gt, GT_PAD, W, H)

  
    bbox_crop, crop_img, crop_orig, mask_full, scale = spotlight_crop_roi(
    str(image_path), instruction
    )

    if bbox_crop is None:
        continue

    bbox_crop=[float(x) for x in bbox_crop]

    prediction, action = agent.predict(instruction, crop_img)
    click_coordinates = extract_click_coordinates(action)

    if click_coordinates is not None:
        px_norm, py_norm = click_coordinates

        cw, ch = crop_img.size
        px_crop_resize = float(px_norm) * cw
        py_crop_resize = float(py_norm) * ch

        px_crop_abs = px_crop_resize / scale
        py_crop_abs = py_crop_resize / scale

        px_abs, py_abs = remap_click(px_crop_abs, py_crop_abs, bbox_crop)

        is_correct = point_in_bbox(px_abs, py_abs, bbox_gt_padded)
    else:
        px_abs=py_abs=None
        is_correct=False

    if is_correct:
        correct+=1

    # vis_img = Image.open(image_path).convert("RGB")
    # draw = ImageDraw.Draw(vis_img)

    # draw.rectangle(bbox_crop, outline="yellow", width=3)
    # draw.rectangle(bbox_gt, outline="green", width=3)

    # if px_abs is not None:
    #     draw.ellipse([px_abs-3,py_abs-3,px_abs+3,py_abs+3], fill="red")

    # vis_img.save(VIS_DIR/f"{sample_id}_{'correct' if is_correct else 'wrong'}.jpg")

    results.append({
        "id": sample_id,
        "gt_bbox": bbox_gt,
        "gt_bbox_padded": bbox_gt_padded,
        "pred_pixel": [px_abs,py_abs] if px_abs is not None else None,
        "correct": is_correct
    })

safe_save()

summary={
    "accuracy": float(correct/len(results)) if results else 0.0
}

print(json.dumps(summary, indent=2))
